In [0]:
import os

os.environ['MLFLOW_DFS_TMP'] = "/Volumes/workspace/default/my_volume/mlflow_tmp"

print("MLflow temporary path set globally!")

In [0]:
import mlflow.spark

model_name = "workspace.default.power_risk_model_production"
alias = "champion"

model_uri = f"models:/{model_name}@{alias}"

loaded_model = mlflow.spark.load_model(model_uri)

print(f"✅ Successfully loaded the current '{alias}' model from Unity Catalog.")

In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F

res = requests.get("https://api.open-meteo.com/v1/forecast?latitude=52.52&longitude=13.41&hourly=temperature_2m,windspeed_10m,cloudcover")
forecast_raw = res.json()['hourly']

pd_df = pd.DataFrame(forecast_raw).head(24) 
pd_df['time'] = pd.to_datetime(pd_df['time'])
live_forecast_df = spark.createDataFrame(pd_df)

live_forecast_df = spark.createDataFrame(pd_df)

forecast_df = live_forecast_df.select(
    F.col("time").cast("timestamp"),
    "temperature_2m",
    "windspeed_10m",
    "cloudcover"
).withColumn("hour", F.hour("time")) \
 .withColumn("month", F.month("time")) \
 .withColumn("day_of_week", F.dayofweek("time")) \
 .withColumn("country", F.lit("DE")) \
 .withColumn("city", F.lit("Berlin"))

forecast_df.show(5)
print("Data types fixed! Ready to predict.")

In [0]:
import mlflow
import mlflow.spark
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import functions as F

run_id = "464b0858b256462c98e4fc0b95574edc"
model_uri = f"runs:/{run_id}/random-forest-model"
volume_temp_path = "/Volumes/workspace/default/my_volume/mlflow_tmp"

loaded_model = mlflow.spark.load_model(
    model_uri=model_uri, 
    dfs_tmpdir=volume_temp_path
)

feature_cols = ["temperature_2m", "windspeed_10m", "cloudcover", "hour", "month", "day_of_week"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
forecast_vector = assembler.transform(forecast_df)

predictions = loaded_model.transform(forecast_vector)

get_risk_prob = F.udf(lambda v: float(v[1]), "float")

final_results = predictions.select(
    "time", 
    "temperature_2m", 
    get_risk_prob("probability").alias("risk_score"),
    "prediction"
).orderBy("time")

display(final_results)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql import functions as F

final_results_with_labels = final_results.withColumn(
    "Alert_Level",
    F.when(F.col("risk_score") > 0.05, "🔴 RED ALERT")
     .when(F.col("risk_score") > 0.03, "🟡 ELEVATED")
     .otherwise("🟢 NORMAL")
)

display(final_results_with_labels.filter(F.col("risk_score") > 0.03))